In [ ]:
import numpy as np

def canonicalize_smiles(smiles):
    """
    Helper function to unify/canonicalize a SMILES string.
    Returns the canonical SMILES if valid, otherwise falls back to the original string.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        pass
    return smiles

def best_mean_scores(cf_mmace, cf_meg):
    best_mean = -np.inf
    best_mmace = None
    best_meg = None

    for i in range(len(cf_mmace['counterfactuals_smiles'])):
        for j in range(len(cf_mmace['counterfactuals_smiles'][i])):

            # --- Get top 3 unique for MMACE ---
            similarity_mmace = np.array(cf_mmace['counterfactuals_similarity'][i][j])
            similarity_mmace_argsort = np.argsort(-similarity_mmace)

            top3_mmace = []
            seen_smiles_mmace = set()
            for idx in similarity_mmace_argsort:
                cf_smile = canonicalize_smiles(cf_mmace['counterfactuals_smiles'][i][j][idx])
                if cf_smile not in seen_smiles_mmace:
                    seen_smiles_mmace.add(cf_smile)
                    top3_mmace.append(idx)
                if len(top3_mmace) == 3:
                    break

            # --- Get top 3 unique for MEG (with reward >= 1) ---
            similarity_meg = np.array(cf_meg['counterfactuals_similarity'][i][j])
            similarity_meg_argsort = np.argsort(-similarity_meg)

            top3_meg = []
            seen_smiles_meg = set()
            for idx in similarity_meg_argsort:
                # Filter by reward score
                if cf_meg['counterfactuals_pred_reward'][i][j][idx] >= 1:
                    cf_smile = canonicalize_smiles(cf_meg['counterfactuals_smiles'][i][j][idx])
                    if cf_smile not in seen_smiles_meg:
                        seen_smiles_meg.add(cf_smile)
                        top3_meg.append(idx)
                if len(top3_meg) == 3:
                    break

            # --- Evaluate if we successfully found 3 valid unique counterfactuals for both ---
            if len(top3_mmace) == 3 and len(top3_meg) == 3:
                mmace_dict = {
                    'smiles': cf_mmace['smiles'][i][j],
                    'counterfactuals': np.array(cf_mmace['counterfactuals_smiles'][i][j])[top3_mmace],
                    'similarities': np.array(cf_mmace['counterfactuals_similarity'][i][j])[top3_mmace],
                }
                meg_dict = {
                    'smiles': cf_meg['smiles'][i][j],
                    'counterfactuals': np.array(cf_meg['counterfactuals_smiles'][i][j])[top3_meg],
                    'similarities': np.array(cf_meg['counterfactuals_similarity'][i][j])[top3_meg],
                }
                mean_top3 = np.mean(mmace_dict['similarities']) + np.mean(meg_dict['similarities'])

                if best_mean < mean_top3:
                    best_mean = mean_top3
                    best_mmace = mmace_dict
                    best_meg = meg_dict

    return best_mmace, best_meg


In [ ]:
datasets_names = {
        'CNOHF': '../results/cnohf_data/cnohf_ecfp/explanations/',
        'Photoswitch': '../results/photoswitch_data/photoswitch_ecfp/explanations/',
        'Polymers': '../results/polymers_data/polymers_ecfp/explanations/',
        'Redox': '../results/redox_data/redox_ecfp/explanations/',
        'COF': '../results/cof_data/cof_ecfp_descriptor/explanations/',
        'Linear': '../results/synthetic_data/herg_ecfp_linear/explanations/',
        'Piecewise': '../results/synthetic_data/herg_ecfp_piecewise/explanations/',
        'Nonlinear': '../results/synthetic_data/herg_ecfp_nonlinear/explanations/',
        'GT Linear': '../results/gt_synthetic_data/herg_ecfp_linear/explanations/',
        'GT Piecewise': '../results/gt_synthetic_data/herg_ecfp_piecewise/explanations/',
        'GT Nonlinear': '../results/gt_synthetic_data/herg_ecfp_nonlinear/explanations/'
    }
meg_cf_name = 'meg2_results.pickle'
mmace_cf_name = 'mmace_results.pickle'

In [ ]:
import pickle
import os

results_mmace = {}
results_meg = {}

for dname, dpath in datasets_names.items():
    with open(os.path.join(dpath, meg_cf_name), 'rb') as f:
        cf_meg = pickle.load(f)
    with open(os.path.join(dpath, mmace_cf_name), 'rb') as f:
        cf_mmace = pickle.load(f)
    mace, meg = best_mean_scores(cf_mmace, cf_meg)
    results_meg[dname] = meg
    results_mmace[dname] = mace

In [ ]:
results_mmace

In [ ]:
results_meg

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import rdFMCS

def visualize_counterfactuals(data_dict, output_filename="counterfactuals_grid.svg", mols_per_row=4):
    """
    Visualizes original molecules and their counterfactuals from a dictionary,
    highlighting the modified/added atoms and bonds.

    Parameters:
    - data_dict (dict): Dictionary containing original SMILES, counterfactuals, and similarities.
    - output_filename (str): The name of the output SVG file (default: "counterfactuals_grid.svg").
    - mols_per_row (int): Number of molecules to display in each row of the grid (default: 4).
    """
    mols = []
    legends = []
    hl_atoms_list = []
    hl_bonds_list = []
    hl_atom_colors = []
    hl_bond_colors = []

    # RGB for highlighting additions (Green)
    green = (102/256, 255/256, 240/256)

    for key, val in data_dict.items():
        orig_mol = Chem.MolFromSmiles(val['smiles'])
        if not orig_mol:
            continue

        # 1. Append the Original Molecule (First Column)
        mols.append(orig_mol)
        legends.append(f"[{key}] Original")
        hl_atoms_list.append([])
        hl_bonds_list.append([])
        hl_atom_colors.append({})
        hl_bond_colors.append({})

        # 2. Append Counterfactuals (Subsequent Columns)
        for cf_smiles, sim in zip(val['counterfactuals'], val['similarities']):
            cf_mol = Chem.MolFromSmiles(cf_smiles)
            if not cf_mol:
                continue

            # Find Maximum Common Substructure (MCS)
            mcs = rdFMCS.FindMCS([orig_mol, cf_mol], timeout=2, matchValences=True)
            mcs_mol = Chem.MolFromSmarts(mcs.smartsString)

            # Match MCS to counterfactual to find what was kept
            match = cf_mol.GetSubstructMatch(mcs_mol)

            # Highlight atoms NOT in the MCS match (these are additions/modifications)
            added_atoms = [atom.GetIdx() for atom in cf_mol.GetAtoms() if atom.GetIdx() not in match]
            added_bonds = [bond.GetIdx() for bond in cf_mol.GetBonds()
                           if not (bond.GetBeginAtomIdx() in match and bond.GetEndAtomIdx() in match)]

            mols.append(cf_mol)
            legends.append(f"Similarity: {sim:.2f}")
            hl_atoms_list.append(added_atoms)
            hl_bonds_list.append(added_bonds)
            hl_atom_colors.append({idx: green for idx in added_atoms})
            hl_bond_colors.append({idx: green for idx in added_bonds})

    # Generate Grid Image
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=mols_per_row,
        subImgSize=(300, 200),
        legends=legends,
        highlightAtomLists=hl_atoms_list,
        highlightBondLists=hl_bonds_list,
        highlightAtomColors=hl_atom_colors,
        highlightBondColors=hl_bond_colors,
        useSVG=True
    )

    # Save the raw SVG string to a file
    svg_data = img.data if hasattr(img, 'data') else img
    with open(output_filename, "w") as f:
        f.write(svg_data)

    print(f"Visualization saved successfully to '{output_filename}'")

In [ ]:
results_meg_gt = {k: results_meg[k] for k in ['Linear', 'Piecewise', 'Nonlinear', 'GT Linear', 'GT Piecewise', 'GT Nonlinear']}
results_meg_rf = {k: results_meg[k] for k in ['CNOHF', 'Photoswitch', 'Polymers', 'Redox', 'COF']}
results_mmace_gt = {k: results_mmace[k] for k in ['Linear', 'Piecewise', 'Nonlinear', 'GT Linear', 'GT Piecewise', 'GT Nonlinear']}
results_mmace_rf = {k: results_mmace[k] for k in ['CNOHF', 'Photoswitch', 'Polymers', 'Redox', 'COF']}

In [ ]:
visualize_counterfactuals(results_mmace_gt, output_filename="counterfactuals_grid_mmace_gt.svg")
visualize_counterfactuals(results_mmace_rf, output_filename="counterfactuals_grid_mmace_rf.svg")

In [ ]:
visualize_counterfactuals(results_meg_gt, output_filename="counterfactuals_grid_meg_gt.svg")
visualize_counterfactuals(results_meg_rf, output_filename="counterfactuals_grid_meg_rf.svg")